# Can Sentiment Predict ADE?

## Prerequisites

In [1]:
!pip install transformers[torch]==4.40.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 89.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.1/775.1 kB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.1 MB/s eta 0:00:00


In [18]:
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import AutoConfig
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import f1_score
import tqdm

## Data

In [3]:
CLEANED_TWEET_FILE="../train_tweets.tsv"
CLEANED_ANNOTATION_FILE="../train_spans_norm.tsv"

In [13]:
tweets = pd.read_csv(CLEANED_TWEET_FILE,
                     sep="\t",
                     header=None,
                     names=["id", "text"])
annotations = pd.read_csv(CLEANED_ANNOTATION_FILE,
                          sep="\t",
                          header=None,
                          names=["id", "label", "start", "end", "ade", "medra"])
tweets["class"] = tweets["id"].isin(annotations["id"])
tweets['class'] = tweets['class'].astype(int)

## Model

See https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest

In [21]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [22]:
config = AutoConfig.from_pretrained(MODEL)

### Experiment

If the model assigns negative label to a tweet then classify it as `1` else `0`.

We keep track of the score.

In [32]:
preds5 = []  # score greater than 0.5
preds6 = []  # score greater than 0.6
preds7 = []  # score greater than 0.7
preds8 = []  # score greater than 0.8
preds9 = []  # score greater than 0.9

In [33]:
model.eval()
for t in tqdm.tqdm(tweets['text'].to_list()):
    encoded_input = tokenizer(t, return_tensors='pt', max_length=512)
    output = model(**encoded_input)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)

    ranking = np.argsort(scores)
    l = config.id2label[ranking[-1]]
    s = scores[ranking[-1]]
    if l != 'negative' or s <= 0.5:
        preds5.append(0)
        preds6.append(0)
        preds7.append(0)
        preds8.append(0)
        preds9.append(0)
        continue
    if s > 0.5:
        preds5.append(1)
    else:
        preds5.append(0)
    if s > 0.6:
        preds6.append(1)
    else:
        preds6.append(0)
    if s > 0.7:
        preds7.append(1)
    else:
        preds7.append(0)
    if s > 0.8:
        preds8.append(1)
    else:
        preds8.append(0)
    if s > 0.9:
        preds9.append(1)
    else:
        preds9.append(0)

100%|██████████| 17190/17190 [23:16<00:00, 12.31it/s]


### F1 score

In [34]:
print("For score > 0.5", f1_score(tweets['class'].values, preds5))
print("For score > 0.6", f1_score(tweets['class'].values, preds6))
print("For score > 0.7", f1_score(tweets['class'].values, preds7))
print("For score > 0.8", f1_score(tweets['class'].values, preds8))
print("For score > 0.9", f1_score(tweets['class'].values, preds9))

For score > 0.5 0.24119501164862273
For score > 0.6 0.25202492211838007
For score > 0.7 0.25909173149990955
For score > 0.8 0.26595744680851063
For score > 0.9 0.23515805705474171
